# 28. Actor-Critic
**Use case:** Financial Services — Monthly Stocks / Bonds / Cash allocation

A wealth manager rebalances balanced portfolios monthly and wants to know whether a learning agent can tilt between growth and defensive mixes better than a static 60/40. Market regimes switch between calm bull markets and stressed bear markets, and nobody announces the switch. Data available: monthly asset returns, rolling volatility and momentum, current holdings and trading costs. Success means a higher Sharpe ratio than 60/40 on unseen periods after costs, with explainable behaviour and a governance trail that model risk can validate.

> 💡 A cricket batter (the actor) chooses shots, while the coach (the critic) watches and says 'that was better or worse than I expected from this position'. The batter does more of the shots that beat the coach's expectation, and the coach keeps refining what to expect.

Legend in code: `# 🐍 PY:` Python lesson · `# 🧠 ML:` ML lesson

In [ ]:
# Run once (Colab already has most of these)
!pip -q install numpy pandas scikit-learn scipy xgboost torch

## CONCEPT: One actor-critic update by hand: Risk-On vs Risk-Off
Perform three actor-critic updates by hand on a two-action allocation choice and watch both the policy and the critic's expectation move.

| 5W + H | |
|---|---|
| **Who** | An investment analyst learning how actor-critic differs from plain trial-and-error. |
| **What** | Two choices (Risk-On 80% stocks, Risk-Off 20% stocks), a critic expecting +0.6% a month and three observed months. |
| **When** | Before touching neural networks: to see the advantage and both update rules in plain numbers. |
| **Where** | A numpy script with a softmax over two preference numbers. |
| **Why** | The whole family (A2C, PPO, SAC) reuses this exact loop: compare outcome with expectation, then update both parts. |
| **How** | Compute TD error = reward - V, push the chosen action's preference by alpha x TD error x grad log pi, nudge V toward the reward. |

**🧠 ML concepts:** Policy as softmax over preferences, Critic value estimate, TD error as advantage, Gradient of log-softmax, Separate actor and critic learning rates  
**🐍 Python concepts:** Dicts keyed by integers, Functions with default arguments, List comprehensions inside f-strings, Array copy vs view, Dict comprehensions with zip, Converting numpy scalars with float()

In [ ]:
# One actor-critic update by hand: should this month's portfolio be "Risk-On" or "Risk-Off"? (Concept)
import numpy as np

# 🐍 PY: a dict maps each action name to the monthly return we happened to observe after choosing it
ACTIONS = ["Risk-On (80% stocks)", "Risk-Off (20% stocks)"]
observed_reward = {0: 1.2, 1: 0.4}  # monthly return in percent: +1.2% vs +0.4%

# 🧠 ML: ACTOR = policy. Two 'preference' numbers turned into probabilities with softmax
theta = np.array([0.0, 0.0])
# 🧠 ML: CRITIC = value estimate V(s): 'how good is a typical month in this market state?'
V = 0.6  # critic currently expects +0.6% a month
alpha_actor, alpha_critic, gamma = 0.5, 0.5, 0.0  # gamma=0 -> one-month episode, no future term

# 🐍 PY: def with a default argument; np.exp works element-wise on the whole array
def softmax(z, temp=1.0):
    e = np.exp((z - z.max()) / temp)
    return e / e.sum()

for month in range(1, 4):
    probs = softmax(theta)
    # 🐍 PY: f-string with format spec :.2f and a list comprehension inside
    print(f"Month {month}: policy probs = {[f'{p:.2f}' for p in probs]}, critic V = {V:.3f}")
    # Pretend the actor sampled action 0 in month 1-2 and action 1 in month 3
    a = 0 if month < 3 else 1
    r = observed_reward[a]
    # 🧠 ML: TD error (advantage) = what happened - what the critic expected
    td_error = r + gamma * 0.0 - V
    print(f"   chose {ACTIONS[a]}, reward = {r:+.1f}%, TD error = {td_error:+.3f}")
    # 🧠 ML: gradient of log softmax = one_hot(a) - probs
    grad_log = -probs.copy()
    grad_log[a] += 1.0
    # 🧠 ML: actor step - push probability UP if advantage > 0, DOWN if < 0
    theta += alpha_actor * td_error * grad_log
    # 🧠 ML: critic step - nudge V toward the observed outcome
    V += alpha_critic * td_error
    print(f"   new theta = {np.round(theta, 3)}, new V = {V:.3f}")

# 🐍 PY: zip pairs names with probabilities; float() turns numpy scalars into plain Python numbers
print("\nFinal policy:", {name: round(float(p), 3) for name, p in zip(ACTIONS, softmax(theta))})
print("Positive surprise raised Risk-On; the below-average Risk-Off month lowered Risk-Off.")

**Try this next:**
- Set the Risk-Off reward to 1.5% in month 3: which way does the policy move now?
- Start the critic at V = 2.0 (very optimistic): what happens to the actor after a +1.2% month?
- Run 20 months alternating actions and print how fast V settles.

## BUILD: A2C in PyTorch choosing among five model portfolios
Train an advantage actor-critic network on regime-switching synthetic markets and compare its allocations with a static 60/40.

| 5W + H | |
|---|---|
| **Who** | A quant research team in an asset manager. |
| **What** | About 100 years of synthetic monthly returns with hidden bull/bear regimes; five model portfolios from 90/10 to 0/30/70. |
| **When** | Early research: does an RL allocator find any signal before investing in a full backtest framework? |
| **Where** | A PyTorch script on CPU; one shared network body with actor and critic heads. |
| **Why** | Static 60/40 ignores regime changes; an agent that reads momentum and volatility may tilt better. |
| **How** | 24-month rollouts, one-step TD advantage, actor loss -log pi x advantage, critic MSE, entropy bonus, then greedy test on 30 unseen years. |

**🧠 ML concepts:** Shared network with two heads, One-step TD advantage, Detaching targets and advantages, Entropy regularisation, Out-of-sample evaluation vs benchmark  
**🐍 Python concepts:** Subclassing nn.Module, List comprehension over a rolling window, torch.distributions.Categorical, Tensor fancy indexing weights[a], lambda functions, enumerate over a list of tuples

In [ ]:
# Advantage Actor-Critic (A2C): monthly Stocks/Bonds/Cash allocation from 5 model portfolios (Build)
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
torch.manual_seed(42)
np.random.seed(42)

# 🐍 PY: list of tuples (stocks, bonds, cash) - the 5 allowed model portfolios = discrete actions
PORTFOLIOS = [(0.9, 0.1, 0.0), (0.7, 0.3, 0.0), (0.5, 0.4, 0.1), (0.3, 0.5, 0.2), (0.0, 0.3, 0.7)]


def simulate_market(n_months):
    """Regime-switching monthly returns: bull markets last ~2 years, bear markets ~8 months."""
    # 🧠 ML: the agent never sees the regime directly - it must infer it from recent returns
    regime, rows = 0, []
    for _ in range(n_months):
        if np.random.rand() < (1 / 24 if regime == 0 else 1 / 8):
            regime = 1 - regime
        mu_s, sd_s = (0.012, 0.035) if regime == 0 else (-0.015, 0.06)
        stock = np.random.normal(mu_s, sd_s)
        bond = np.random.normal(0.003 if regime == 0 else 0.006, 0.012)
        rows.append((stock, bond, 0.002))
    return np.array(rows)


def make_states(rets):
    # 🐍 PY: list comprehension building features from a rolling window of the last 3 and 6 months
    feats = [[rets[t - 3:t, 0].sum(), rets[t - 6:t, 0].std(), rets[t - 3:t, 1].sum(), rets[t - 1, 0]]
             for t in range(6, len(rets))]
    return torch.tensor(np.array(feats) * 10, dtype=torch.float32), rets[6:]


class ActorCritic(nn.Module):
    # 🐍 PY: a class that inherits nn.Module; __init__ builds layers, forward defines the computation
    def __init__(self, n_in, n_act):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_in, 32), nn.Tanh())
        # 🧠 ML: two heads share one body: the ACTOR outputs action logits, the CRITIC outputs V(s)
        self.actor = nn.Linear(32, n_act)
        self.critic = nn.Linear(32, 1)

    def forward(self, x):
        h = self.body(x)
        return self.actor(h), self.critic(h).squeeze(-1)


train_s, train_r = make_states(simulate_market(1206))  # ~100 years of synthetic months
test_s, test_r = make_states(simulate_market(366))     # 30 unseen years
weights = torch.tensor(PORTFOLIOS, dtype=torch.float32)
model = ActorCritic(4, len(PORTFOLIOS))
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
gamma, T = 0.5, 24  # learn from 24-month rollouts

for epoch in range(1, 801):
    start = np.random.randint(0, len(train_s) - T - 1)
    s = train_s[start:start + T + 1]
    logits, values = model(s)
    # 🧠 ML: Categorical turns logits into a probability distribution we can sample from
    dist = torch.distributions.Categorical(logits=logits[:-1])
    a = dist.sample()
    # 🧠 ML: reward = portfolio return in % (weights . asset returns for that month)
    r = (weights[a] * torch.tensor(train_r[start:start + T], dtype=torch.float32)).sum(1) * 100
    # 🧠 ML: one-step TD target and advantage = r + gamma*V(s') - V(s)
    target = r + gamma * values[1:].detach()
    adv = target - values[:-1]
    actor_loss = -(dist.log_prob(a) * adv.detach()).mean()
    critic_loss = adv.pow(2).mean()
    loss = actor_loss + 0.5 * critic_loss - 0.01 * dist.entropy().mean()
    opt.zero_grad()
    loss.backward()
    opt.step()
    if epoch % 160 == 0:
        print(f"epoch {epoch:3d}  avg reward {r.mean():+.3f}%/mo  critic loss {critic_loss.item():.3f}")

# 🧠 ML: evaluate greedily on unseen months versus a static 60/40 stocks/bonds portfolio
with torch.no_grad():
    acts = model(test_s)[0].argmax(1)
agent = (weights[acts].numpy() * test_r).sum(1)
bench = test_r @ np.array([0.6, 0.4, 0.0])
# 🐍 PY: a small helper defined with lambda for annualised return and volatility
ann = lambda x: (x.mean() * 12 * 100, x.std() * np.sqrt(12) * 100)
print("\nTest period (30 years)   ann_return  ann_vol")
for name, x in [("A2C agent", agent), ("60/40 static", bench)]:
    print(f"{name:22s} {ann(x)[0]:9.2f}% {ann(x)[1]:7.2f}%")
print("\nHow often each model portfolio was chosen:")
for i, w in enumerate(PORTFOLIOS):
    print(f"  stocks {w[0]:.0%} / bonds {w[1]:.0%} / cash {w[2]:.0%}: {(acts == i).float().mean():.0%}")
print("Business read: the agent mostly holds the growth mix and switches to the defensive mix in some months.")
print("It earned more than 60/40 here but with more volatility - the Production level adds risk and cost penalties.")

**Try this next:**
- Set gamma to 0.9: does the critic loss get noisier and does the policy change?
- Remove the entropy term and see whether the policy collapses to one portfolio early.
- Add 'last month's bond return' as a fifth feature and compare test results.

## PRODUCTION: Risk-adjusted A2C with costs, tuning and Sharpe vs 60/40
Build a production-style A2C allocator with transaction costs, a mean-variance reward tuned on validation data, and a test-period Sharpe comparison against 60/40.

| 5W + H | |
|---|---|
| **Who** | A quant team presenting an RL allocator to the investment committee and model risk management. |
| **What** | Separate train (100y), validation (30y) and test (40y) synthetic markets, holdings-aware state, 10 bps costs, five model portfolios. |
| **When** | Pre-pilot validation: one shot at the test set after tuning on validation. |
| **Where** | An offline research pipeline; the chosen model and its settings are saved with torch.save. |
| **Why** | Return alone rewards reckless risk; costs punish flip-flopping. Sharpe after costs is what clients and committees care about. |
| **How** | Batched 16-episode rollouts, n-step returns, gradient clipping, lambda grid on validation Sharpe, test-period Sharpe, drawdown and turnover vs 60/40 and 90/10. |

**🧠 ML concepts:** Risk-adjusted (mean-variance) reward, Transaction costs in the state and reward, n-step returns with bootstrap, Hyperparameter tuning on a validation period, Sharpe ratio and max drawdown, Model risk and suitability constraints  
**🐍 Python concepts:** Classes with helper methods, np.stack and np.tile for batching, zip(*...) to unzip results, Closures returning policy functions, Dict of name -> function, Nested format specs in f-strings

In [ ]:
# A2C portfolio allocator with risk-adjusted reward, transaction costs and Sharpe vs 60/40 (Production)
import os
import tempfile
import numpy as np
import torch
import torch.nn as nn

torch.set_num_threads(1)
SEED = 42
torch.manual_seed(SEED)
PORTFOLIOS = np.array([(0.9, 0.1, 0.0), (0.7, 0.3, 0.0), (0.6, 0.4, 0.0), (0.4, 0.5, 0.1), (0.1, 0.4, 0.5)])
TC = 0.0010  # 10 bps per unit of turnover (commissions + spread + market impact)


def simulate_market(n, rng):
    """Regime-switching returns for stocks, bonds, cash. Bear regimes: stocks fall, bonds rally."""
    regime, out = 0, np.zeros((n, 3))
    for t in range(n):
        if rng.rand() < (1 / 30 if regime == 0 else 1 / 9):
            regime = 1 - regime
        s = rng.normal(0.010, 0.035) if regime == 0 else rng.normal(-0.012, 0.060)
        b = rng.normal(0.003, 0.012) if regime == 0 else rng.normal(0.006, 0.015)
        out[t] = (s, b, 0.002)
    return out


class PortfolioEnv:
    """State = market features + current holdings, so the agent can 'see' its transaction cost."""
    def __init__(self, rets, risk_aversion):
        self.rets, self.lam = rets, risk_aversion

    def features(self, t, w):
        r = self.rets
        # 🐍 PY: np.concatenate glues the market signals and the current weights into one vector
        mkt = [r[t - 3:t, 0].sum(), r[t - 6:t, 0].std(), r[t - 12:t, 0].sum(), r[t - 3:t, 1].sum(), r[t - 1, 0]]
        return np.concatenate([np.array(mkt) * 10, w]).astype(np.float32)

    def batch_features(self, ts, W):
        # 🐍 PY: np.stack builds a (batch, 8) matrix so the network scores many episodes in one call
        return torch.from_numpy(np.stack([self.features(t, w) for t, w in zip(ts, W)]))

    def step(self, t, w_old, a):
        w_new = PORTFOLIOS[a]
        # 🧠 ML: transaction cost = turnover x cost rate; discourages flip-flopping every month
        cost = TC * np.abs(w_new - w_old).sum()
        port = w_new @ self.rets[t] - cost
        # 🧠 ML: risk-adjusted reward = return - lambda * squared deviation (a monthly mean-variance utility)
        reward = port - self.lam * (port - 0.005) ** 2
        return reward * 100, port, w_new


class ActorCritic(nn.Module):
    def __init__(self, n_in, n_act, hidden=64):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(n_in, hidden), nn.Tanh(), nn.Linear(hidden, hidden), nn.Tanh())
        self.actor, self.critic = nn.Linear(hidden, n_act), nn.Linear(hidden, 1)

    def forward(self, x):
        h = self.body(x)
        return self.actor(h), self.critic(h).squeeze(-1)


def train(rets, lam, epochs=150, T=24, B=16, gamma=0.6, lr=3e-3, seed=SEED):
    torch.manual_seed(seed)
    rng, env = np.random.RandomState(seed), PortfolioEnv(rets, lam)
    model = ActorCritic(8, len(PORTFOLIOS))
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        # 🧠 ML: B parallel episodes (different start months) = a less noisy gradient per update
        t0 = rng.randint(12, len(rets) - T - 1, size=B)
        W = np.tile(PORTFOLIOS[2], (B, 1))
        logps, vals, rews, ents = [], [], [], []
        # 🧠 ML: roll out T months; holdings carry over, so the actor's choice changes the next state
        for k in range(T):
            logits, v = model(env.batch_features(t0 + k, W))
            dist = torch.distributions.Categorical(logits=logits)
            a = dist.sample().numpy()
            # 🐍 PY: zip over three arrays at once, then unpack the list of tuples with zip(*...)
            r, _, W = map(np.array, zip(*[env.step(t, w, ai) for t, w, ai in zip(t0 + k, W, a)]))
            logps.append(dist.log_prob(torch.from_numpy(a)))
            vals.append(v)
            rews.append(torch.tensor(r, dtype=torch.float32))
            ents.append(dist.entropy())
        with torch.no_grad():
            R = model(env.batch_features(t0 + T, W))[1]
        # 🧠 ML: n-step returns computed backwards from the critic's bootstrap value of the final state
        rets_n = []
        for r in reversed(rews):
            R = r + gamma * R
            rets_n.insert(0, R)
        rets_n, vals, logps = torch.stack(rets_n), torch.stack(vals), torch.stack(logps)
        # 🧠 ML: advantage = n-step return - critic's estimate; detach so the actor doesn't train the critic
        adv = rets_n - vals
        loss = -(logps * adv.detach()).mean() + 0.5 * adv.pow(2).mean() - 0.01 * torch.stack(ents).mean()
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    return model


def backtest(rets, policy, lam=0.0):
    env, w, port, turns = PortfolioEnv(rets, lam), PORTFOLIOS[2], [], []
    for t in range(12, len(rets)):
        a = policy(env.features(t, w), w)
        turns.append(np.abs(PORTFOLIOS[a] - w).sum())
        _, p, w = env.step(t, w, a)
        port.append(p)
    p = np.array(port)
    # 🧠 ML: Sharpe = excess return over cash / volatility, annualised (x sqrt(12) for monthly data)
    sharpe = (p.mean() - 0.002) / p.std() * np.sqrt(12)
    dd = (np.cumprod(1 + p) / np.maximum.accumulate(np.cumprod(1 + p)) - 1).min()
    return {"ann_ret%": p.mean() * 1200, "ann_vol%": p.std() * np.sqrt(12) * 100, "sharpe": sharpe,
            "max_dd%": dd * 100, "turnover/yr": np.mean(turns) * 12}


def greedy(model):
    def pol(x, w):
        with torch.no_grad():
            return int(model(torch.from_numpy(x))[0].argmax())
    return pol


def main():
    rng = np.random.RandomState(SEED)
    train_r, valid_r, test_r = simulate_market(1200, rng), simulate_market(360, rng), simulate_market(480, rng)
    # 🧠 ML: tune the risk-aversion lambda on a VALIDATION period; test period is touched only once
    print("Tuning risk aversion (validation Sharpe):")
    best = None
    for lam in (0.0, 3.0, 8.0):
        m = train(train_r, lam)
        s = backtest(valid_r, greedy(m))["sharpe"]
        print(f"  lambda={lam:3.1f}  valid Sharpe={s:5.2f}")
        if best is None or s > best[1]:
            best = (lam, s, m)
    lam, _, model = best
    print(f"Chosen lambda = {lam}")

    # 🐍 PY: dict of name -> policy function; lambdas ignore the state for the static baselines
    policies = {"A2C agent": greedy(model), "60/40 static": lambda x, w: 2, "Always 90/10": lambda x, w: 0}
    print("\nTest period (40 unseen years), after 10 bps transaction costs:")
    print(f"{'strategy':14s} {'ann_ret%':>9s} {'ann_vol%':>9s} {'sharpe':>7s} {'max_dd%':>8s} {'turnover/yr':>12s}")
    for name, pol in policies.items():
        m = backtest(test_r, pol)
        print(f"{name:14s} " + " ".join(f"{m[k]:{9 if 'ann' in k else 7 if k == 'sharpe' else 8 if k == 'max_dd%' else 12}.2f}"
                                       for k in m))

    # 🧠 ML: explainability - how does the actor's probability mass shift from calm to stressed markets?
    w0 = PORTFOLIOS[2]
    calm = np.concatenate([np.array([0.03, 0.02, 0.12, 0.01, 0.01]) * 10, w0]).astype(np.float32)
    stress = np.concatenate([np.array([-0.10, 0.07, -0.15, 0.03, -0.05]) * 10, w0]).astype(np.float32)
    for label, x in [("calm uptrend", calm), ("stressed selloff", stress)]:
        with torch.no_grad():
            probs = torch.softmax(model(torch.from_numpy(x))[0], -1).numpy()
        # 🐍 PY: np.round on the whole probability vector; one probability per model portfolio
        print(f"{label:17s} P(90/10, 70/30, 60/40, 40/50/10, 10/40/50) = {np.round(probs, 2)}")

    path = os.path.join(tempfile.mkdtemp(), "a2c_allocator.pt")
    torch.save({"state_dict": model.state_dict(), "lambda": lam, "portfolios": PORTFOLIOS.tolist()}, path)
    print(f"\nSaved model to {path}")
    print("MODEL-RISK NOTE (SR 11-7): synthetic regimes are not history. Validate on real out-of-sample data,")
    print("stress-test 2008/2020/2022-style scenarios, cap allocation changes, and keep a human PM sign-off.")
    print("Suitability (Reg BI / fiduciary duty) still applies: the agent cannot know a client's risk tolerance.")


if __name__ == "__main__":
    main()

**Try this next:**
- Set TC to 0.0050 (50 bps) and see how turnover and Sharpe change.
- Force lambda = 8.0 and compare drawdown with the chosen model.
- Add a sixth portfolio of 100% cash and check whether the agent uses it in the stress scenario.